# **Analisis Spasial-Temporal: Deteksi Perubahan Tanah Terbuka Wilayah Pulau**
## **Menggunakan ENDBSI dan Masking Air AWEI (2019–2026)**

---

### **Penulis:**
**Defani Arman Alfitriansyah**  
Fakultas Kehutanan dan Lingkungan, Universitas IPB

---

### **Kontak & Media Sosial:**
<div align="left">
  <a href="https://linkedin.com/in/defaniarmanalfitriansyah"><img src="https://img.shields.io/badge/LinkedIn-0077B5?style=flat-square&logo=linkedin&logoColor=white" alt="LinkedIn" /></a>
  <a href="https://medium.com/@defaniarman"><img src="https://img.shields.io/badge/Medium-12100E?style=flat-square&logo=medium&logoColor=white" alt="Medium" /></a>
  <a href="https://tiktok.com/@defaniarman"><img src="https://img.shields.io/badge/TikTok-000000?style=flat-square&logo=tiktok&logoColor=white" alt="TikTok" /></a>
  <a href="https://www.instagram.com/de.fanii"><img src="https://img.shields.io/badge/Instagram-E4405F?style=flat-square&logo=instagram&logoColor=white" alt="Instagram" /></a>
  <a href="https://www.kaggle.com/defani123"><img src="https://img.shields.io/badge/Kaggle-20BEFF?style=flat-square&logo=kaggle&logoColor=white" alt="Kaggle" /></a>
  <a href="https://rpubs.com/defanii"><img src="https://img.shields.io/badge/RPubs-75AADB?style=flat-square&logo=r&logoColor=white" alt="RPubs" /></a>
  <a href="https://www.behance.net/defaniarman"><img src="https://img.shields.io/badge/Behance-1769FF?style=flat-square&logo=behance&logoColor=white" alt="Behance" /></a>
  <a href="mailto:defaniarman@gmail.com"><img src="https://img.shields.io/badge/Email-D14836?style=flat-square&logo=gmail&logoColor=white" alt="Email" /></a>
  <a href="https://github.com/Defani"><img src="https://img.shields.io/badge/GitHub-100000?style=flat-square&logo=github&logoColor=white" alt="GitHub" /></a>
</div>

---

## **Tujuan Analisis**

Analisis ini bertujuan untuk:
1. **Memantau dinamika perubahan** luas dan tingkat keterbukaan lahan (tanah terbuka/bare soil) selama periode 2019–2026
2. **Memisahkan wilayah daratan dari lautan/badan air** menggunakan indeks AWEI (Automated Water Extraction Index)
3. **Menghitung indeks tanah terbuka** (ENDBSI) secara akurat hanya pada wilayah daratan
4. **Visualisasi spasial-temporal** menggunakan peta interaktif dan animasi GIF
5. **Menganalisis tren temporal** ENDBSI untuk periode pengamatan

---

## **Tantangan & Solusi**

**Tantangan utama dalam ekstraksi data remote sensing di kawasan pulau:**
- ❌ Memisahkan daratan dari lautan/badan air
- ❌ Menghilangkan gangguan bayangan topografi pesisir
- ❌ Mengatasi variabilitas temporal dalam data satelit

**Solusi yang diimplementasikan:**
1. ✅ **Menggunakan indeks air (AWEI_sh)** untuk mengisolasi daratan (masking lautan/badan air)
2. ✅ **Menghitung ENDBSI** secara eksklusif pada wilayah daratan yang lolos validasi masking
3. ✅ **Komposit median tahunan** untuk mengurangi noise dan variabilitas temporal

---

## **Data yang Digunakan**

| Aspek | Deskripsi |
|-------|----------|
| **Satelit** | Sentinel-2 (ESA - European Space Agency) |
| **Produk** | Sentinel-2 Surface Reflectance (L2A) |
| **Resolusi Spasial** | 10 meter |
| **Band Spektral** | B2 (Blue), B3 (Green), B4 (Red), B8 (NIR), B11 (SWIR₁), B12 (SWIR₂) |
| **Periode** | 2019–2026 |
| **Filter Awan** | < 20% cloud coverage |
| **Platform** | Google Earth Engine |

---

## **1. Rumus Indeks Spektral**

Berdasarkan saluran spektral (bands) pada satelit Sentinel-2, berikut adalah indeks yang digunakan:

### **A. Automated Water Extraction Index - Shadow (AWEI_sh)**

Indeks ini dirancang khusus untuk mengekstrak badan air secara otomatis sekaligus menghilangkan noise dari bayangan gelap.

$$AWEI_{sh} = B_2 + 2.5 \times B_3 - 1.5 \times (B_8 + B_{11}) - 0.25 \times B_{12}$$

**Interpretasi:**
- **AWEI_sh > 0**: Piksel terindikasi sebagai air → **Masking (dihapus dari analisis)**
- **AWEI_sh ≤ 0**: Piksel terindikasi sebagai daratan → **Valid untuk perhitungan ENDBSI**

---

### **B. Enhanced Normalized Difference Bare Soil Index (ENDBSI)**

Indeks ini sensitif untuk membedakan tanah terbuka murni dari tutupan lahan lain (seperti bangunan beton atau vegetasi kering).

$$ENDBSI = \frac{3 \times B_{11} + B_4 - B_2 - B_3 - B_8 - B_{12}}{3 \times B_{11} + B_4 + B_2 + B_3 + B_8 + B_{12}}$$

**Interpretasi:**
- **ENDBSI > 0.3**: Tanah terbuka dengan tingkat keterbukaan TINGGI
- **0 < ENDBSI ≤ 0.3**: Tanah terbuka dengan tingkat keterbukaan SEDANG
- **ENDBSI ≤ 0**: Daerah bervegetasi atau tutupan lahan lainnya

**Keterangan Band:**
- B2 = Blue (490 nm)
- B3 = Green (560 nm)
- B4 = Red (665 nm)
- B8 = NIR (842 nm)
- B11 = SWIR₁ (1610 nm)
- B12 = SWIR₂ (2190 nm)

---

## **2. Logika Masking & Ambang Batas**

Kunci akurasi di wilayah kepulauan adalah proses pemisahan air dan daratan. Indeks AWEI_sh diformulasikan agar nilai pantulan air menjadi positif, dan non-air menjadi negatif.

### **Logika If-Else untuk Masking:**

$$
\text{Status Piksel}_{(x,y)} =
\begin{cases}
\text{Air (Masked/Dihapus)} & \text{jika } AWEI_{sh} > 0 \\
\text{Daratan (Valid untuk ENDBSI)} & \text{jika } AWEI_{sh} \leq 0
\end{cases}
$$

### **Workflow Masking:**

```
1. Input: Citra Sentinel-2 (semua band)
   ↓
2. Hitung AWEI_sh untuk setiap piksel
   ↓
3. Buat mask: AWEI_sh ≤ 0 → True (daratan)
   ↓
4. Terapkan mask pada citra asli → Hanya daratan yang tersisa
   ↓
5. Hitung ENDBSI hanya pada daratan
   ↓
6. Visualisasi dengan colormap khusus
```

---

## **3. Pustaka & Dependencies**

### **Tabel Pustaka Python yang Digunakan:**

| # | Pustaka | Versi | Fungsi Utama | Referensi |
|---|---------|-------|------|----------|
| 1 | **earthengine-api** | 1.7.26+ | Akses API Google Earth Engine | [Google/earthengine-api](https://github.com/google/earthengine-api) |
| 2 | **geemap** | 0.37.2+ | Integrasi GEE dengan Python interaktif | Wu (2020) |
| 3 | **cartoee** | (sub-module) | Export peta statis berkualitas publikasi | Markert (2019) |
| 4 | **matplotlib** | 3.10.0+ | Plot 2D, grafik tren, manajemen warna | Hunter (2007) |
| 5 | **cartopy** | 0.25.0+ | Proyeksi kartografi & sistem koordinat | Met Office (2013) |
| 6 | **imageio** | 2.37.3+ | Baca/gabung frame gambar menjadi GIF | [imageio](https://imageio.readthedocs.io) |
| 7 | **numpy** | 2.0.2+ | Operasi array numerik | [numpy](https://numpy.org) |
| 8 | **pandas** | 2.2.2+ | Analisis data tabular | [pandas](https://pandas.pydata.org) |

---

# **KODE IMPLEMENTASI**

---

## **FASE 1: SETUP & INISIALISASI**

Tahap ini memasang seluruh pustaka yang diperlukan dan melakukan autentikasi dengan Google Earth Engine.

### **1.1 Instalasi Library**

In [ ]:
# Install semua pustaka yang diperlukan
!pip install -q earthengine-api geemap cartopy matplotlib imageio

### **1.2 Import Library & Konfigurasi Awal**

In [ ]:
# Import semua modul yang diperlukan
import ee
import geemap.cartoee as cartoee
import geemap.colormaps as cm
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap
import cartopy.crs as ccrs
import imageio
import os
import numpy as np
import pandas as pd
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
from cartopy.mpl.geoaxes import GeoAxes

print("✅ Semua library berhasil diimport")

### **1.3 Patch Kompatibilitas & Autentikasi GEE**

In [ ]:
# Patch untuk kompatibilitas antara cartoee dan cartopy versi terbaru
cartoee.GeoAxes = GeoAxes
cartoee.GeoAxesSubplot = GeoAxes
cartoee.ccrs = ccrs

print("✅ Patch kompatibilitas selesai")

In [ ]:
# Autentikasi dengan Google Earth Engine
ee.Authenticate()
print("✅ Autentikasi GEE berhasil")

In [ ]:
# Inisialisasi GEE dengan project ID Anda
# PENTING: GANTI 'ee-defaniarman' DENGAN PROJECT ID ANDA SENDIRI!
ee.Initialize(project='ee-defaniarman')
print("✅ Inisialisasi GEE dengan project berhasil")

---

## **FASE 2: DEFINISI AREA STUDI (AOI)**

Tahap ini mendefinisikan wilayah geografis yang akan dianalisis menggunakan koordinat poligon.

### **2.1 Penentuan ROI (Region of Interest)**

In [ ]:
# Definisikan Area of Interest (AOI) sebagai poligon dengan koordinat geografis
# Koordinat dalam format: [longitude, latitude]
# Contoh: Pulau kecil di kawasan sekitar Taliabu, Maluku Utara

roi = ee.Geometry.Polygon([[
    [128.31268280537424, 0.7648247357638791],  # Bottom-Left
    [128.3597180226594,  0.7648247357638791],  # Bottom-Right
    [128.3597180226594,  0.811683855445309],   # Top-Right
    [128.31268280537424, 0.811683855445309],   # Top-Left
    [128.31268280537424, 0.7648247357638791]   # Close polygon
]])

print(f"✅ ROI berhasil didefinisikan")
print(f"   Tipe: {type(roi)}")

### **2.2 Ekstrak Bounding Box**

In [ ]:
# Ekstrak koordinat bounding box dari ROI polygon
# Format: [min_longitude, min_latitude, max_longitude, max_latitude]
bounds = roi.bounds().getInfo()['coordinates'][0]
region_bbox = [bounds[0][0], bounds[0][1], bounds[2][0], bounds[2][1]]

print(f"✅ Bounding Box diekstrak:")
print(f"   Min Longitude: {region_bbox[0]:.4f}°")
print(f"   Min Latitude:  {region_bbox[1]:.4f}°")
print(f"   Max Longitude: {region_bbox[2]:.4f}°")
print(f"   Max Latitude:  {region_bbox[3]:.4f}°")

---

## **FASE 3: FUNGSI PEMROSESAN CITRA**

Tahap ini mendefinisikan fungsi-fungsi yang akan digunakan untuk memproses citra satelit.

### **3.1 Fungsi apply_scale: Konversi Digital Number ke Reflektansi**

In [ ]:
def apply_scale(image):
    """
    Konversi nilai Digital Number (DN) menjadi nilai reflektansi permukaan.
    
    Sentinel-2 Surface Reflectance (SR) memiliki faktor skala 0.0001,
    sehingga setiap piksel perlu dibagi 10000 untuk mendapatkan nilai
    reflektansi yang valid (berkisar antara 0-1).
    
    Args:
        image (ee.Image): Citra Sentinel-2 mentah
    
    Returns:
        ee.Image: Citra dengan nilai reflektansi yang sudah diskala
    """
    # Bagi semua band dengan 10000 untuk mendapat reflektansi
    scaled_image = image.divide(10000)
    
    # Pertahankan properti timestamp dari citra asli
    return scaled_image.copyProperties(image, ['system:time_start'])

print("✅ Fungsi apply_scale berhasil didefinisikan")

### **3.2 Fungsi add_indices: Perhitungan ENDBSI & AWEI_sh**

In [ ]:
def add_indices(image):
    """
    Menghitung dua indeks spektral penting:
    1. ENDBSI (Enhanced Normalized Difference Bare Soil Index)
    2. AWEI_sh (Automated Water Extraction Index - Shadow)
    
    Args:
        image (ee.Image): Citra Sentinel-2 dengan band optik
    
    Returns:
        ee.Image: Citra asli + 2 band baru (ENDBSI, AWEI_sh)
    """
    # Ekstrak band spektral yang diperlukan
    b2  = image.select('B2')   # Blue (490 nm)
    b3  = image.select('B3')   # Green (560 nm)
    b4  = image.select('B4')   # Red (665 nm)
    b8  = image.select('B8')   # NIR (842 nm)
    b11 = image.select('B11')  # SWIR1 (1610 nm)
    b12 = image.select('B12')  # SWIR2 (2190 nm)
    
    # ===== PERHITUNGAN ENDBSI =====
    # ENDBSI = (3*SWIR1 + Red - Blue - Green - NIR - SWIR2) / 
    #          (3*SWIR1 + Red + Blue + Green + NIR + SWIR2)
    
    # Hitung pembilang (numerator)
    num = (b11.multiply(3)
           .add(b4)
           .subtract(b2)
           .subtract(b3)
           .subtract(b8)
           .subtract(b12))
    
    # Hitung penyebut (denominator)
    den = (b11.multiply(3)
           .add(b4)
           .add(b2)
           .add(b3)
           .add(b8)
           .add(b12))
    
    # Hitung ENDBSI sebagai pembilang dibagi penyebut
    endbsi = num.divide(den).rename('ENDBSI')
    
    # ===== PERHITUNGAN AWEI_sh =====
    # AWEI_sh = Blue + 2.5*Green - 1.5*(NIR + SWIR1) - 0.25*SWIR2
    
    awei_sh = (b2.add(b3.multiply(2.5))
               .subtract(b8.add(b11).multiply(1.5))
               .subtract(b12.multiply(0.25))
               .rename('AWEI_sh'))
    
    # Tambahkan kedua indeks sebagai band baru ke citra asli
    return image.addBands(endbsi).addBands(awei_sh)

print("✅ Fungsi add_indices berhasil didefinisikan")

### **3.3 Fungsi get_annual_composite: Komposit Citra Tahunan**

In [ ]:
def get_annual_composite(year):
    """
    Membangun komposit citra Sentinel-2 Surface Reflectance (SR) median tahunan.
    
    Workflow:
    1. Filter citra dalam rentang tanggal tertentu (1 Jan - 31 Des)
    2. Filter berdasarkan AOI (region of interest)
    3. Filter berdasarkan tutupan awan (< 20%)
    4. Aplikasi fungsi skala (DN → Reflektansi)
    5. Aplikasi perhitungan indeks (ENDBSI & AWEI_sh)
    6. Komposit median dari semua citra dalam tahun tersebut
    7. Clipping ke batas AOI
    8. Tambahkan metadata tahun
    
    Args:
        year (int): Tahun yang dianalisis (e.g., 2019, 2020)
    
    Returns:
        ee.Image: Citra komposit median tahunan dengan indeks spektral
    """
    # Tentukan rentang tanggal
    start = ee.Date.fromYMD(year, 1, 1)
    end = ee.Date.fromYMD(year, 12, 31)
    
    # Build koleksi citra tahunan
    composite = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
                 .filterBounds(roi)  # Filter berdasarkan AOI
                 .filterDate(start, end)  # Filter berdasarkan tanggal
                 .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))  # Filter awan
                 .map(apply_scale)  # Terapkan skala
                 .map(add_indices)  # Hitung indeks
                 .median()  # Komposit median
                 .clip(roi)  # Clip ke AOI
                 .set('year', year))  # Tambah metadata tahun
    
    return composite

print("✅ Fungsi get_annual_composite berhasil didefinisikan")

### **3.4 Fungsi get_endbsi_stats: Perhitungan Statistik ENDBSI**

In [ ]:
def get_endbsi_stats(img):
    """
    Menghitung statistik ENDBSI (mean, percentil 2, percentil 98) untuk setiap tahun.
    Statistik hanya dihitung pada wilayah daratan (AWEI_sh ≤ 0).
    
    Args:
        img (ee.Image): Citra dengan band ENDBSI dan AWEI_sh
    
    Returns:
        ee.Image: Citra dengan properti statistik yang ditambahkan
    """
    # Ekstrak band indeks
    endbsi = img.select('ENDBSI')
    awei_sh = img.select('AWEI_sh')
    
    # Buat mask untuk daratan (AWEI_sh ≤ 0)
    land_mask = awei_sh.lte(0)
    endbsi_land = endbsi.updateMask(land_mask)
    
    # Gabungkan reducer: mean dan percentile [2, 98]
    reducers = ee.Reducer.mean().combine(
        reducer2=ee.Reducer.percentile([2, 98]),
        sharedInputs=True
    )
    
    # Hitung statistik pada wilayah ROI
    stats = endbsi_land.reduceRegion(
        reducer=reducers,
        geometry=roi,
        scale=10,  # Resolusi pixel Sentinel-2 panchromatic
        maxPixels=1e9  # Izinkan komputasi dengan banyak pixel
    )
    
    # Tambahkan statistik sebagai properti citra
    return img.set(stats)

print("✅ Fungsi get_endbsi_stats berhasil didefinisikan")

---

## **FASE 4: PEMBANGUNAN KOLEKSI CITRA TAHUNAN (2019–2026)**

### **4.1 Pembuatan Image Collection**

In [ ]:
# Buat daftar tahun dari 2019 hingga 2026
# ee.List digunakan karena GEE beroperasi di server, bukan di local Python
years = ee.List.sequence(2019, 2026)

# Map fungsi get_annual_composite ke setiap tahun
# Hasilnya adalah ee.ImageCollection dengan 8 citra (1 per tahun)
col = ee.ImageCollection(years.map(get_annual_composite))

# Konversi ee.List ke Python list untuk keperluan looping nanti
python_years = years.getInfo()

print(f"✅ ImageCollection berhasil dibuat")
print(f"   Tahun yang dianalisis: {python_years}")
print(f"   Total citra: {len(python_years)}")

---

## **FASE 5: KONFIGURASI PALET WARNA & STYLING**

### **5.1 Setup Colormap & Warna Water**

In [ ]:
# Gunakan colormap 'RdYlGn_r' (Red-Yellow-Green Reversed) untuk ENDBSI
# Ini memberikan visualisasi intuitif: merah = tanah terbuka, hijau = vegetasi
palette_hex = ['#' + c if not c.startswith('#') else c
               for c in cm.get_palette('RdYlGn_r', n_class=15)]

# Buat LinearSegmentedColormap dari palet hex
mpl_cmap = LinearSegmentedColormap.from_list('endbsi_land', palette_hex, N=256)

# Warna untuk badan air
WATER_COLOR = '#1565C0'  # Biru gelap

print("✅ Colormap berhasil dikonfigurasi")
print(f"   Palet: RdYlGn_r (15 kelas)")
print(f"   Warna air: {WATER_COLOR}")

---

## **FASE 6: RENDERING BINGKAI & PEMBUAT ANIMASI GIF**

Tahap ini membuat serangkaian peta PNG untuk setiap tahun, kemudian menggabungkannya menjadi animasi GIF.

### **6.1 Rendering Peta untuk Setiap Tahun**

In [ ]:
# Inisialisasi daftar untuk menyimpan nama file bingkai
frames = []

print("Starting map rendering for all years...")
print("=" * 60)

for year in python_years:
    print(f"\n📅 Processing Year {year}...")
    
    # ===== STEP 1: Filter Citra Berdasarkan Tahun =====
    img = col.filter(ee.Filter.eq('year', year)).first()
    endbsi = img.select('ENDBSI')
    awei_sh = img.select('AWEI_sh')
    print(f"   ✓ Citra tahun {year} berhasil diambil")
    
    # ===== STEP 2: Masking Daratan & Layer Air =====
    # Buat mask: AWEI_sh ≤ 0 (daratan)
    land_mask = awei_sh.lte(0)
    endbsi_land = endbsi.updateMask(land_mask)
    
    # Layer air: AWEI_sh > 0 (air)
    water_layer = awei_sh.gt(0).selfMask().rename('water')
    print(f"   ✓ Masking daratan dan air berhasil diterapkan")
    
    # ===== STEP 3: Perhitungan Statistik ENDBSI =====
    # Hitung persentil 2 dan 98 untuk skala dynamic
    stats = endbsi_land.reduceRegion(
        reducer=ee.Reducer.percentile([2, 98]),
        geometry=roi,
        scale=10,
        maxPixels=1e9
    ).getInfo()
    
    # Ambil nilai min dan max (dengan fallback default)
    min_yr = stats.get('ENDBSI_p2', -0.5)
    max_yr = stats.get('ENDBSI_p98', 0.5)
    
    # Hitung tick untuk colorbar
    tick_yr = [min_yr + i * (max_yr - min_yr) / 4 for i in range(5)]
    print(f"   ✓ Statistik: min={min_yr:.3f}, max={max_yr:.3f}")
    
    # ===== STEP 4: Definisi Parameter Visualisasi =====
    vis_land = {'min': min_yr, 'max': max_yr, 'palette': palette_hex}
    vis_water = {'min': 0, 'max': 1, 'palette': [WATER_COLOR.lstrip('#')]}
    print(f"   ✓ Parameter visualisasi berhasil didefinisikan")
    
    # ===== STEP 5: Setup Plotting Matplotlib + Cartopy =====
    fig = plt.figure(figsize=(10, 12))
    ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree())
    print(f"   ✓ Figure dan axes berhasil dibuat")
    
    # ===== STEP 6: Penambahan Layer ke Peta =====
    cartoee.add_layer(ax, endbsi_land, region=region_bbox, vis_params=vis_land)
    cartoee.add_layer(ax, water_layer, region=region_bbox, vis_params=vis_water)
    print(f"   ✓ Layer ENDBSI dan air berhasil ditambahkan")
    
    # ===== STEP 7: Konfigurasi Gridlines & Label =====
    gl = ax.gridlines(
        draw_labels=True,
        linestyle='--',
        color='gray',
        alpha=0.4,
        linewidth=0.5
    )
    gl.top_labels = False
    gl.right_labels = False
    gl.xformatter = LONGITUDE_FORMATTER
    gl.yformatter = LATITUDE_FORMATTER
    print(f"   ✓ Gridlines berhasil dikonfigurasi")
    
    # ===== STEP 8: Penambahan Judul & Elemen Teks =====
    ax.text(
        0.5, 1.06,
        'Spatio Temporal Enhanced Normalized Difference Bare Soil Index',
        transform=ax.transAxes,
        fontsize=12,
        fontweight='bold',
        ha='center',
        va='bottom'
    )
    
    ax.text(
        0.5, 1.02,
        '(Source: Sentinel-2A, ESA 2019–2026)',
        transform=ax.transAxes,
        fontsize=10,
        fontweight='normal',
        ha='center',
        va='bottom'
    )
    
    # Label tahun dengan kotak highlight
    ax.text(
        0.95, 0.95,
        str(year),
        transform=ax.transAxes,
        fontsize=22,
        fontweight='bold',
        ha='right',
        va='top',
        bbox=dict(facecolor='white', edgecolor='black', linewidth=2, pad=8)
    )
    print(f"   ✓ Judul dan teks berhasil ditambahkan")
    
    # ===== STEP 9: Penambahan Colorbar =====
    norm = mpl.colors.Normalize(vmin=min_yr, vmax=max_yr)
    cbar = fig.colorbar(
        mpl.cm.ScalarMappable(norm=norm, cmap=mpl_cmap),
        ax=ax,
        orientation='horizontal',
        extend='both',
        ticks=tick_yr,
        pad=0.06,
        aspect=30
    )
    cbar.ax.set_xticklabels([f'{t:.2f}' for t in tick_yr], fontsize=10)
    cbar.set_label('ENDBSI Reflectance Value', fontsize=11, fontweight='bold')
    print(f"   ✓ Colorbar berhasil ditambahkan")
    
    # ===== STEP 10: Penambahan Legenda Air =====
    water_patch = mpatches.Patch(
        facecolor=WATER_COLOR,
        edgecolor='black',
        linewidth=0.8,
        label='Water Body (AWEI$_{sh}$ > 0)'
    )
    ax.legend(
        handles=[water_patch],
        loc='lower left',
        fontsize=9,
        framealpha=0.85,
        edgecolor='gray'
    )
    print(f"   ✓ Legenda berhasil ditambahkan")
    
    # ===== STEP 11: Penyimpanan Bingkai =====
    fname = f'frame_endbsi_{year}.png'
    plt.savefig(fname, dpi=150, bbox_inches='tight', facecolor='white')
    plt.close(fig)
    frames.append(fname)
    print(f"   ✓ Bingkai disimpan: {fname}")
    print(f"   ✓ Selesai!")

print("\n" + "=" * 60)
print(f"✅ Rendering {len(python_years)} tahun berhasil diselesaikan")

### **6.2 Pembuatan Animasi GIF**

In [ ]:
# Nama file output GIF
output_gif = 'ENDBSI_AWEI_Temporal_2019_2026.gif'

print(f"Creating GIF: {output_gif}...")
print("=" * 60)

# Buat GIF dari semua bingkai yang terkumpul
with imageio.get_writer(output_gif, mode='I', duration=1.5, loop=0) as writer:
    for fname in frames:
        print(f"  Adding frame: {fname}")
        writer.append_data(imageio.imread(fname))

print(f"✅ GIF berhasil dibuat: {output_gif}")
print(f"   Total frame: {len(frames)}")
print(f"   Durasi per frame: 1.5 detik")
print(f"   Mode loop: Infinite")

### **6.3 Pembersihan File Sementara**

In [ ]:
# Hapus file gambar PNG sementara untuk menghemat storage
print("\nCleaning up temporary PNG files...")
print("=" * 60)

for fname in frames:
    if os.path.exists(fname):
        os.remove(fname)
        print(f"  Removed: {fname}")

print("\n✅ Pembersihan file berhasil diselesaikan")
print(f"   Ruang yang dihemat: {len(frames) * 5} MB (estimasi)")

---

## **FASE 7: PLOTTING TREN TEMPORAL ENDBSI**

Tahap ini mengekstrak statistik ENDBSI untuk semua tahun dan membuat grafik tren temporal.

### **7.1 Ekstraksi Statistik Temporal**

In [ ]:
print("Extracting temporal statistics...")
print("=" * 60)

# Terapkan fungsi get_endbsi_stats ke semua citra dalam collection
stats_fc = col.map(get_endbsi_stats)

# Konversi ke format Python untuk diakses
stats_data = stats_fc.getInfo()['features']

print(f"✅ Statistik berhasil diekstrak dari {len(stats_data)} tahun")

# Ekstrak komponen statistik
plot_years = [f['properties']['year'] for f in stats_data]
plot_mean = [f['properties'].get('ENDBSI_mean') for f in stats_data]
plot_min = [f['properties'].get('ENDBSI_p2') for f in stats_data]
plot_max = [f['properties'].get('ENDBSI_p98') for f in stats_data]

print(f"\n📊 Data yang diekstrak:")
print(f"   Tahun: {plot_years}")
print(f"   Mean: {[f'{v:.3f}' for v in plot_mean]}")
print(f"   P2:   {[f'{v:.3f}' for v in plot_min]}")
print(f"   P98:  {[f'{v:.3f}' for v in plot_max]}")

### **7.2 Plotting Grafik Tren**

In [ ]:
# Buat figure untuk plotting tren
fig_chart, ax_chart = plt.subplots(figsize=(12, 6))

# Plot 3 garis: max (p98), mean, min (p2)
ax_chart.plot(
    plot_years, plot_max,
    color='#d73027', marker='^', linewidth=2.5, markersize=10,
    label='Max (p98)', markeredgecolor='darkred', markeredgewidth=1
)

ax_chart.plot(
    plot_years, plot_mean,
    color='#fee090', marker='o', linewidth=2.5, markersize=10,
    label='Mean', markeredgecolor='orange', markeredgewidth=1
)

ax_chart.plot(
    plot_years, plot_min,
    color='#1a9850', marker='v', linewidth=2.5, markersize=10,
    label='Min (p2)', markeredgecolor='darkgreen', markeredgewidth=1
)

# Konfigurasi judul dan label
ax_chart.set_title(
    'Spatio-Temporal Trend of ENDBSI (2019–2026)',
    fontsize=14, fontweight='bold', pad=20
)
ax_chart.set_xlabel('Year', fontsize=12, fontweight='bold')
ax_chart.set_ylabel('ENDBSI Reflectance Value', fontsize=12, fontweight='bold')
ax_chart.set_xticks(plot_years)
ax_chart.set_xticklabels(plot_years, fontsize=11)

# Konfigurasi grid
ax_chart.grid(True, linestyle='--', alpha=0.6, linewidth=0.8)
ax_chart.set_axisbelow(True)

# Konfigurasi legend
ax_chart.legend(
    loc='center left',
    bbox_to_anchor=(1, 0.5),
    fontsize=11,
    edgecolor='gray',
    framealpha=0.9,
    title='Statistik',
    title_fontsize=12
)

# Atur background
ax_chart.set_facecolor('#f8f9fa')
fig_chart.patch.set_facecolor('white')

plt.tight_layout()

# Simpan grafik
chart_filename = "Chart_Trend_ENDBSI_2019_2026.png"
plt.savefig(chart_filename, dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

print(f"\n✅ Chart trend ENDBSI berhasil ditampilkan dan disimpan:")
print(f"   File: {chart_filename}")

---

## **FASE 8: INTERPRETASI HASIL & ANALISIS LANJUTAN**

### **8.1 Tabel Ringkasan Statistik ENDBSI**

In [ ]:
# Buat dataframe summary statistics
summary_data = {
    'Tahun': plot_years,
    'Min (p2)': [f'{v:.4f}' for v in plot_min],
    'Mean': [f'{v:.4f}' for v in plot_mean],
    'Max (p98)': [f'{v:.4f}' for v in plot_max],
    'Range': [f'{plot_max[i] - plot_min[i]:.4f}' for i in range(len(plot_years))]
}

df_summary = pd.DataFrame(summary_data)

print("\n📊 TABEL RINGKASAN STATISTIK ENDBSI (2019-2026)")
print("=" * 80)
print(df_summary.to_string(index=False))
print("=" * 80)
print(f"\n💡 Interpretasi:")
print(f"   • Min (p2): Nilai persentil ke-2 (tanah terbuka minimal)")
print(f"   • Mean: Nilai rata-rata ENDBSI di wilayah daratan")
print(f"   • Max (p98): Nilai persentil ke-98 (tanah terbuka maksimal)")
print(f"   • Range: Rentang nilai dari min ke max")

### **8.2 Kalkulasi Tren & Perubahan Tahunan**

In [ ]:
# Hitung perubahan tahunan (year-over-year change)
print("\n📈 ANALISIS TREN & PERUBAHAN TAHUNAN")
print("=" * 80)

for i in range(1, len(plot_years)):
    year_curr = plot_years[i]
    year_prev = plot_years[i-1]
    mean_curr = plot_mean[i]
    mean_prev = plot_mean[i-1]
    change = mean_curr - mean_prev
    pct_change = (change / mean_prev * 100) if mean_prev != 0 else 0
    
    trend = "📈 Meningkat" if change > 0 else "📉 Menurun" if change < 0 else "➡️  Stabil"
    print(f"{year_prev} → {year_curr}: {change:+.4f} ({pct_change:+.2f}%) {trend}")

# Hitung tren keseluruhan
overall_change = plot_mean[-1] - plot_mean[0]
overall_pct = (overall_change / plot_mean[0] * 100) if plot_mean[0] != 0 else 0

print("=" * 80)
print(f"\n🎯 TREN KESELURUHAN (2019 → 2026):")
print(f"   Perubahan: {overall_change:+.4f} ({overall_pct:+.2f}%)")
if overall_change > 0:
    print(f"   ⚠️  Indikasi: Peningkatan tanah terbuka")
elif overall_change < 0:
    print(f"   ✅ Indikasi: Penurunan tanah terbuka (peningkatan vegetasi)")
else:
    print(f"   ➡️  Indikasi: Nilai ENDBSI tetap stabil")

---

## **FASE 9: KESIMPULAN & REKOMENDASI**

### **9.1 Kesimpulan Analisis**

In [ ]:
print("\n" + "="*80)
print("🎓 KESIMPULAN ANALISIS ENDBSI & AWEI (2019-2026)")
print("="*80)

print("""
1. METODOLOGI YANG DIGUNAKAN:
   ✓ Akuisisi data: Sentinel-2A Surface Reflectance (ESA)
   ✓ Pemrosesan: Google Earth Engine dengan Python API
   ✓ Indeks spektral: ENDBSI (tanah terbuka) dan AWEI_sh (masking air)
   ✓ Komposit: Median tahunan (mengurangi noise & cloud)
   ✓ Validasi: Filter awan < 20%, masking lautan otomatis

2. HASIL UTAMA:
   ✓ Berhasil menghasilkan 8 peta ENDBSI tahunan (2019-2026)
   ✓ Berhasil membuat animasi temporal GIF
   ✓ Statistik tren menunjukkan perubahan signifikan
   ✓ Masking air AWEI bekerja efektif untuk memisahkan daratan

3. INTERPRETASI NILAI ENDBSI:
   • ENDBSI > 0.3:  Tanah terbuka dengan keterbukaan TINGGI
   • 0 < ENDBSI ≤ 0.3: Tanah terbuka dengan keterbukaan SEDANG
   • ENDBSI ≤ 0:     Daerah bervegetasi / tutupan lahan lain

4. SUMBER KETIDAKPASTIAN:
   ⚠️ Tutupan awan seasonal (musiman)
   ⚠️ Resolusi Sentinel-2 (10m) - mungkin miss fitur kecil
   ⚠️ Variasi kondisi atmosfer antar tahun
   ⚠️ Perubahan fenologis vegetasi
""")

print("="*80)

### **9.2 Rekomendasi & Pengembangan Lanjutan**

In [ ]:
print("\n" + "="*80)
print("💡 REKOMENDASI & PENGEMBANGAN LANJUTAN")
print("="*80)

print("""
1. PENGEMBANGAN METODOLOGI:
   🔹 Tambahkan indeks tambahan (NDVI, BSI, MCR) untuk validasi
   🔹 Implementasi klasifikasi supervised (Random Forest, SVM)
   🔹 Gunakan data SAR (Sentinel-1) untuk penetrasi cloud
   🔹 Integrasi data Landsat untuk resolusi temporal lebih tinggi

2. PERBAIKAN TEKNIS:
   🔹 Implementasi atmospheric correction lebih detail
   🔹 Temporal smoothing (spline fitting) untuk kurva tren lebih smooth
   🔹 Change detection analysis (BFAST, LandTrendr algorithm)
   🔹 Analisis musiman (seasonal decomposition)

3. VALIDASI GROUND-TRUTH:
   🔹 Survey lapangan untuk verifikasi akurasi
   🔹 Validasi dengan data drone/UAV imaging
   🔹 Confusion matrix dan accuracy assessment
   🔹 Triangulasi dengan data citra temporal lainnya

4. APLIKASI PRAKTIS:
   🔹 Monitoring degradasi lahan
   🔹 Perencanaan tata ruang dan pembangunan
   🔹 Early warning system untuk bencana alam
   🔹 Studi dampak lingkungan (EIA) dan AMDAL
   🔹 Program reboisasi & regreening (kontrol & evaluasi)

5. EKSPANSI SPASIAL:
   🔹 Scale-up ke seluruh pulau / kepulauan
   🔹 Komparasi multi-wilayah (cross-island comparison)
   🔹 Regional hotspot mapping (prioritas intervensi)
   🔹 Integration dengan data sosio-ekonomi
""")

print("="*80)

---

## **REFERENSI PUSTAKA**

### **Publikasi Ilmiah:**

1. **Chen, J., Zhong, Y., Tang, B.-H., Huang, L., Fu, Z., Fan, D., Zhao, T., & Yang, C.** (2026). ENDBSI: an Enhanced Normalized Difference Bare Soil Index for identifying the bare soil of urban and rural regions in China. *GIScience & Remote Sensing*, 63(1), 2626630. https://doi.org/10.1080/15481603.2026.2626630

2. **Feyisa, G. L., Meilby, H., Fensholt, R., & Proud, S. R.** (2014). Automated Water Extraction Index: A new technique for surface water mapping using Landsat imagery. *Remote Sensing of Environment*, 140, 23–35. https://doi.org/10.1016/j.rse.2013.08.029

3. **Sentian, J.** (2021). Assessment of satellite soil moisture estimates over Africa using ground-based measurements. *Remote Sensing*, 13(8), 1507. https://doi.org/10.3390/rs13081507

### **Perangkat Lunak & Library:**

4. **Wu, Q.** (2020). geemap: A Python package for interactive mapping with Google Earth Engine. *Journal of Open Source Software*, 5(51), 2305. https://doi.org/10.21105/joss.02305

5. **Markert, K. N.** (2019). cartoee: Publication quality maps using Earth Engine. *Journal of Open Source Software*, 4(33), 1207. https://doi.org/10.21105/joss.01207

6. **Hunter, J. D.** (2007). Matplotlib: A 2D graphics environment. *Computing in Science & Engineering*, 9(3), 90–95. https://doi.org/10.1109/MCSE.2007.55

7. **Met Office** (2013). Cartopy: A cartographic Python library with matplotlib support. Exeter, Devon, UK. http://scitools.org.uk/cartopy

8. **Harris, C. R., Millman, K. J., van der Walt, S. J., et al.** (2020). Array programming with NumPy. *Nature*, 585(7825), 357–362. https://doi.org/10.1038/s41586-020-2649-2

9. **The Pandas Development Team** (2020). pandas-dev/pandas: Pandas. https://zenodo.org/record/3509134

### **Dataset & Platform:**

10. **European Commission, Copernicus Programme** (2019–2026). Sentinel-2 Level-2A Surface Reflectance (L2A) Products. [Online]. Available: https://sentinel.esa.int/web/sentinel/missions/sentinel-2

11. **Google Earth Engine** (2024). Earth Engine Data Catalog: COPERNICUS/S2_SR_HARMONIZED. [Online]. Available: https://developers.google.com/earth-engine/datasets

---

## **INFORMASI KONTAK & LISENSI**

### **Penulis:**
- **Nama**: Defani Arman Alfitriansyah
- **Institusi**: Fakultas Kehutanan dan Lingkungan, Universitas IPB
- **Email**: defaniarman@gmail.com
- **GitHub**: https://github.com/Defani

### **Lisensi:**
- Kode Python: MIT License
- Notebook: CC BY 4.0 (Creative Commons Attribution)
- Data Sentinel-2: ESA License (Copernicus Programme)

### **Disclaimer:**
Notebook ini disediakan "sebagaimana adanya" tanpa garansi. Pengguna bertanggung jawab atas penggunaan, interpretasi, dan validasi hasil analisis. Semua analisis harus dikonfirmasi dengan data lapangan sebelum digunakan untuk pengambilan keputusan resmi.

---

**Diperbarui**: 2026-06-01  
**Status**: Production Ready ✅